# Decode the trained NMT model and get the BLEU score on the test set

Generating the outputs on the test set, for the models trained with varying vocab sizes (21K, 30K, 40K, 50K, 60K).

* Import model binaries and artifacts from GCS bucket using `gcsfuse` 
* Use a Colab GPU Instance to run the decode script

In [1]:
from google.colab import auth
auth.authenticate_user()

!echo "deb http://packages.cloud.google.com/apt gcsfuse-$(lsb_release -c -s) main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!apt-get update
!apt-get install gcsfuse

!mkdir -p /tmp/bucket/
!gcsfuse japanese_english_nmt /tmp/bucket

!mkdir -p /tmp/vocab_bucket/
!gcsfuse bucket-for-everything-in-vertex /tmp/vocab_bucket

deb http://packages.cloud.google.com/apt gcsfuse-jammy main
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1022  100  1022    0     0   8667      0 --:--:-- --:--:-- --:--:--  8735
OK
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://cli.github.com/packages stable InRelease                         
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://packages.cloud.google.com/apt gcsfuse-jammy InRelease [1,227 B]   
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:9 https://developer.download.nvidia

In [2]:
!pip install docopt sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 641.5 kB/s eta 0:00:000:00:01
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=8d806b316eaeb2686e8a77763d106eca19ea930aaea2cdf36dd423339dced368
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [3]:
import os
os.listdir('/tmp/bucket/models/')

['model_21k.bin',
 'model_21k.bin.optim',
 'model_30k.bin',
 'model_30k.bin.optim',
 'model_40k.bin',
 'model_40k.bin.optim',
 'model_50k.bin',
 'model_50k.bin.optim',
 'model_60k.bin',
 'model_60k.bin.optim']

In [4]:
os.chdir('/tmp/vocab_bucket/ja_en_nmt/python_package')
os.listdir('/tmp/vocab_bucket/ja_en_nmt/python_package')

['.ipynb_checkpoints',
 'Dockerfile',
 'README.md',
 '__pycache__',
 'create_job.sh',
 'download_data.py',
 'run.py',
 'run.sh',
 'setup.py',
 'src_21k.model',
 'src_21k.vocab',
 'src_30k.model',
 'src_30k.vocab',
 'src_40k.model',
 'src_40k.vocab',
 'src_50k.model',
 'src_50k.vocab',
 'src_60k.model',
 'src_60k.vocab',
 'tgt.model',
 'tgt.vocab',
 'trainer',
 'vocab.py',
 'vocab_21k.json',
 'vocab_30k.json',
 'vocab_40k.json',
 'vocab_50k.json',
 'vocab_60k.json']

## Decoding with 21K Vocab Model

In [9]:
!ARG_COMPILE=--${2:-'no-compile'}
!ARG_BACKEND=--backend=${3:-'inductor'}

!python run.py decode \
  /tmp/bucket/models/model_21k.bin \
  /tmp/bucket/test.ja \
  /tmp/bucket/test.en \
  ../test_outputs_21k.txt \
  --src-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/src_21k \
  --tgt-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/tgt \
  $ARG_COMPILE $ARG_BACKEND --gpu

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
{'--backend': 'inductor',
 '--batch-size': '32',
 '--beam-size': '5',
 '--clip-grad': '5.0',
 '--compile': False,
 '--dev-src': None,
 '--dev-tgt': None,
 '--dropout': '0.3',
 '--embed-size': '256',
 '--gpu': True,
 '--help': False,
 '--hidden-size': '256',
 '--input-feed': False,
 '--log-every': '10',
 '--lr': '0.001',
 '--lr-decay': '0.5',
 '--max-decoding-time-step': '70',
 '--max-epoch': '30',
 '--max-num-trial': '5',
 '--no-compile': False,
 '--patience': '5',
 '--sample-size': '5',
 '--save-to': 'model.bin',
 '--seed': '0',
 '--src-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/src_21k',
 '--tgt-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/tgt',
 '--train-src': None,
 '--train-tgt': None,
 '--uniform-init': '0.1',
 '--valid-niter': '2000',
 '--vocab': None,
 '--vocab-size-src': '21000',
 'MODEL_PATH': '/tmp/bucket/models/model_21k.bin',
 '

## Decoding with 30K Vocab Model

In [6]:
!ARG_COMPILE=--${2:-'no-compile'}
!ARG_BACKEND=--backend=${3:-'inductor'}

!CUDA_VISIBLE_DEVICES=0 python run.py decode \
  /tmp/bucket/models/model_30k.bin \
  /tmp/bucket/test.ja \
  /tmp/bucket/test.en \
  ../test_outputs_30k.txt \
  --src-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/src_30k \
  --tgt-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/tgt \
  $ARG_COMPILE $ARG_BACKEND --gpu

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
{'--backend': 'inductor',
 '--batch-size': '32',
 '--beam-size': '5',
 '--clip-grad': '5.0',
 '--compile': False,
 '--dev-src': None,
 '--dev-tgt': None,
 '--dropout': '0.3',
 '--embed-size': '256',
 '--gpu': True,
 '--help': False,
 '--hidden-size': '256',
 '--input-feed': False,
 '--log-every': '10',
 '--lr': '0.001',
 '--lr-decay': '0.5',
 '--max-decoding-time-step': '70',
 '--max-epoch': '30',
 '--max-num-trial': '5',
 '--no-compile': False,
 '--patience': '5',
 '--sample-size': '5',
 '--save-to': 'model.bin',
 '--seed': '0',
 '--src-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/src_30k',
 '--tgt-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/tgt',
 '--train-src': None,
 '--train-tgt': None,
 '--uniform-init': '0.1',
 '--valid-niter': '2000',
 '--vocab': None,
 '--vocab-size-src': '21000',
 'MODEL_PATH': '/tmp/bucket/models/model_30k.bin',
 '

## Decoding with 40K Vocab Model

In [5]:
!ARG_COMPILE=--${2:-'no-compile'}
!ARG_BACKEND=--backend=${3:-'inductor'}

!CUDA_VISIBLE_DEVICES=0 python run.py decode \
  /tmp/bucket/models/model_40k.bin \
  /tmp/bucket/test.ja \
  /tmp/bucket/test.en \
  ../test_outputs_40k.txt \
  --src-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/src_40k \
  --tgt-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/tgt \
  $ARG_COMPILE $ARG_BACKEND --gpu

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
{'--backend': 'inductor',
 '--batch-size': '32',
 '--beam-size': '5',
 '--clip-grad': '5.0',
 '--compile': False,
 '--dev-src': None,
 '--dev-tgt': None,
 '--dropout': '0.3',
 '--embed-size': '256',
 '--gpu': True,
 '--help': False,
 '--hidden-size': '256',
 '--input-feed': False,
 '--log-every': '10',
 '--lr': '0.001',
 '--lr-decay': '0.5',
 '--max-decoding-time-step': '70',
 '--max-epoch': '30',
 '--max-num-trial': '5',
 '--no-compile': False,
 '--patience': '5',
 '--sample-size': '5',
 '--save-to': 'model.bin',
 '--seed': '0',
 '--src-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/src_40k',
 '--tgt-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/tgt',
 '--train-src': None,
 '--train-tgt': None,
 '--uniform-init': '0.1',
 '--valid-niter': '2000',
 '--vocab': None,
 '--vocab-size-src': '21000',
 'MODEL_PATH': '/tmp/bucket/models/model_40k.bin',
 'OUTPU

## Decoding with 50K Vocab Model

In [8]:
import torch
torch.cuda.is_available()

False

In [7]:
!ARG_COMPILE=--${2:-'no-compile'}
!ARG_BACKEND=--backend=${3:-'inductor'}

!CUDA_VISIBLE_DEVICES=0 python run.py decode \
  /tmp/bucket/models/model_50k.bin \
  /tmp/bucket/test.ja \
  /tmp/bucket/test.en \
  ../test_outputs_50k.txt \
  --src-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/src_50k \
  --tgt-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/tgt \
  $ARG_COMPILE $ARG_BACKEND --gpu

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
{'--backend': 'inductor',
 '--batch-size': '32',
 '--beam-size': '5',
 '--clip-grad': '5.0',
 '--compile': False,
 '--dev-src': None,
 '--dev-tgt': None,
 '--dropout': '0.3',
 '--embed-size': '256',
 '--gpu': True,
 '--help': False,
 '--hidden-size': '256',
 '--input-feed': False,
 '--log-every': '10',
 '--lr': '0.001',
 '--lr-decay': '0.5',
 '--max-decoding-time-step': '70',
 '--max-epoch': '30',
 '--max-num-trial': '5',
 '--no-compile': False,
 '--patience': '5',
 '--sample-size': '5',
 '--save-to': 'model.bin',
 '--seed': '0',
 '--src-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/src_50k',
 '--tgt-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/tgt',
 '--train-src': None,
 '--train-tgt': None,
 '--uniform-init': '0.1',
 '--valid-niter': '2000',
 '--vocab': None,
 '--vocab-size-src': '21000',
 'MODEL_PATH': '/tmp/bucket/models/model_50k.bin',
 '

## Decoding with 60K Vocab Model

In [5]:
!ARG_COMPILE=--${2:-'no-compile'}
!ARG_BACKEND=--backend=${3:-'inductor'}

!CUDA_VISIBLE_DEVICES=0 python run.py decode \
  /tmp/bucket/models/model_60k.bin \
  /tmp/bucket/test.ja \
  /tmp/bucket/test.en \
  ../test_outputs_60k.txt \
  --src-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/src_60k \
  --tgt-vocab-model=/tmp/vocab_bucket/ja_en_nmt/python_package/tgt \
  $ARG_COMPILE $ARG_BACKEND --gpu

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
{'--backend': 'inductor',
 '--batch-size': '32',
 '--beam-size': '5',
 '--clip-grad': '5.0',
 '--compile': False,
 '--dev-src': None,
 '--dev-tgt': None,
 '--dropout': '0.3',
 '--embed-size': '256',
 '--gpu': True,
 '--help': False,
 '--hidden-size': '256',
 '--input-feed': False,
 '--log-every': '10',
 '--lr': '0.001',
 '--lr-decay': '0.5',
 '--max-decoding-time-step': '70',
 '--max-epoch': '30',
 '--max-num-trial': '5',
 '--no-compile': False,
 '--patience': '5',
 '--sample-size': '5',
 '--save-to': 'model.bin',
 '--seed': '0',
 '--src-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/src_60k',
 '--tgt-vocab-model': '/tmp/vocab_bucket/ja_en_nmt/python_package/tgt',
 '--train-src': None,
 '--train-tgt': None,
 '--uniform-init': '0.1',
 '--valid-niter': '2000',
 '--vocab': None,
 '--vocab-size-src': '21000',
 'MODEL_PATH': '/tmp/bucket/models/model_60k.bin',
 'OUTPU

# Plotting

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(
    {'vocab_size': ['21K', '30K', '40K', '50K', '60K'],
     'BLEU_test': [19.58 , 19.207, 19.726, 19.556, 19.404]
    }
)

df